In [1]:
import geopandas as gpd
import osmnx as ox
import pathlib 
NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_DIRECTORY = NOTEBOOK_PATH / "data"
OUTPUT_DIRECTORY = NOTEBOOK_PATH / "docs"
NOTEBOOK_PATH

PosixPath('/home/jovyan/my-work/practical4')

In [2]:
flood_yhdistetty = gpd.read_file(DATA_DIRECTORY/"tulva_yhdistetty.gpkg")
flood_area = gpd.read_file(DATA_DIRECTORY/"tulva_kohtalainen"/"Tulva_kohtalainen.shp")

In [3]:
G = ox.io.load_graphml(OUTPUT_DIRECTORY/"turku_bike_network.graphml")
nodes, edges = ox.graph_to_gdfs(G)

In [4]:

# Project both to EPSG:3067
edges_projected = edges.to_crs(3067).reset_index()
flood_projected = flood_area.to_crs(3067)

# Keep only the necessary columns
edges_for_overlay = edges_projected[
    ["u", "v", "key", "geometry"]
]

flood_for_overlay = flood_projected[["geometry"]]

In [13]:
# Sections of cycling edges inside the flood extent
flooded_segments = gpd.overlay(
    edges_for_overlay,
    flood_for_overlay,
    how="intersection",
    keep_geom_type=True
)

flooded_segments["flooded"] = True
# Calculate length of flooded portions
flooded_segments["flooded_length_m"] = (
    flooded_segments.geometry.length
)
flooded_lengths = (
    flooded_segments
    .groupby(["u", "v", "key"])["flooded_length_m"]
    .sum()
)

# Sections of cycling edges outside the flood extent
dry_segments = gpd.overlay(
    edges_for_overlay,
    flood_for_overlay,
    how="difference",
    keep_geom_type=True
)

dry_segments["flooded"] = False

In [14]:
split_edges = gpd.GeoDataFrame(
    pd.concat(
        [flooded_segments, dry_segments],
        ignore_index=True
    ),
    crs=edges_for_overlay.crs
)

split_edges["segment_length_m"] = split_edges.geometry.length

print(split_edges["flooded"].value_counts())

flooded
True     65525
False    55088
Name: count, dtype: int64


In [15]:
# edges retains its original (u, v, key) MultiIndex
edges["flooded_length_m"] = flooded_lengths

# Set non-intersecting edges to zero
edges["flooded_length_m"] = (
    edges["flooded_length_m"].fillna(0)
)

# Mark edges as flooded if intersection length > 0
edges["flooded"] = edges["flooded_length_m"] > 0

print(edges[["flooded", "flooded_length_m"]].head())

                        flooded  flooded_length_m
u      v           key                           
224093 13313453370 0       True          8.738582
224097 5272039635  0      False          0.000000
       166809249   0      False          0.000000
       1616825494  0      False          0.000000
224101 530809840   0       True          6.371555


KeyError: 'Column not found: flooded_length_m'

In [16]:
# First use the dissolved flood extent for accurate
# total flooded lengths without overlapping polygon duplicates.

# Total flooded length per original edge
flooded_lengths = (
    flooded_segments
    .groupby(["u", "v", "key"])["flooded_length_m"]
    .sum()
)

# Original edges and their geometric lengths
edge_stats = edges_for_overlay.copy()

edge_stats["total_length_m"] = (
    edge_stats.geometry.length
)

edge_stats = edge_stats.set_index(["u", "v", "key"])

edge_stats["flooded_length_m"] = (
    flooded_lengths
    .reindex(edge_stats.index)
    .fillna(0)
)

edge_stats["flooded_fraction"] = (
    edge_stats["flooded_length_m"]
    / edge_stats["total_length_m"]
)

# Edges that are partly, but not completely, flooded
partly_flooded = edge_stats[
    (edge_stats["flooded_fraction"] > 0) &
    (edge_stats["flooded_fraction"] < 1)
]

print("Partly flooded edges:", len(partly_flooded))

print(
    partly_flooded[
        ["total_length_m", "flooded_length_m",
         "flooded_fraction"]
    ].describe()
)

Partly flooded edges: 16408
       total_length_m  flooded_length_m  flooded_fraction
count    16408.000000      16408.000000      16408.000000
mean        92.405661         22.376524          0.317210
std         99.552747         36.987523          0.295688
min          1.214320          0.001771          0.000067
25%         28.821870          4.038759          0.068827
50%         62.365726          9.775759          0.213564
75%        120.717845         25.942909          0.507211
max       1594.125409        893.411505          1.000000


In [17]:
import pandas as pd

bins = [0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 1.01]

counts = pd.cut(
    partly_flooded["flooded_fraction"],
    bins=bins
).value_counts().sort_index()

print(counts)

flooded_fraction
(0.0, 0.01]      674
(0.01, 0.05]    2535
(0.05, 0.1]     2146
(0.1, 0.25]     3531
(0.25, 0.5]     3358
(0.5, 0.75]     2079
(0.75, 1.01]    2085
Name: count, dtype: int64


In [ ]:
G_floodinfo = G.copy()

for (u, v, key), row in edges.iterrows():
    G_floodinfo[u][v][key]["flooded"] = bool(row["flooded"])
    G_floodinfo[u][v][key]["flooded_length_m"] = float(
        row["flooded_length_m"]
    )

ox.save_graphml(
    G_floodinfo,
    filepath="turku_bike_network_floodinfo_lengths.graphml"
)

In [ ]:
G_floodinfo = G.copy()

for (u, v, key), row in edges.iterrows():
    G_floodinfo[u][v][key]["flooded"] = bool(row["flooded"])
    G_floodinfo[u][v][key]["flooded_length_m"] = float(
        row["flooded_length_m"]
    )

ox.save_graphml(
    G_floodinfo,
    filepath="turku_bike_network_floodinfo_lengths.graphml"
)

In [ ]:
# Convert back to GeoDataFrame
_, edges_check = ox.graph_to_gdfs(G_floodinfo)

m = edges_check.explore(
    column="flooded",
    categorical=True,
    cmap="RdYlGn_r",
    legend=True,
    
)

m